# Thermal detector training - YOLOv8n on the FLIR ADAS train split

**Purpose.** Train a YOLOv8n detector on FLIR thermal imagery so the model-comparison notebook
(`IR_SR_model_comparison`, Section 11) can test whether the super-resolution ranking depends on the
detector. The comparison itself is not performed here.

**Output.** `checkpoints/yolov8n_flir_thermal.pt` under the project Drive root, plus a JSON report in
`presentation_assets/`.

## Important operating notes
1. **Source directory.** Training reads `train/thermal_8_bit` only. The sibling directory
   `train/Annotated_thermal_8_bit` contains the same frames with bounding boxes drawn on the pixels;
   a detector trained on it learns to detect rectangles and scores near zero on clean imagery.
   Cell 2 asserts the correct directory and cell 3 lets you verify visually before training.
2. **Leakage.** The 1,366-image evaluation set of the comparison notebook must not appear in training.
   Cell 2 asserts zero overlap and aborts otherwise. The internal validation split is carved from the
   training set, never from the evaluation set.
3. **Runtime.** About 1.7 h for 50 epochs on a T4. Safe to leave unattended: checkpoints are written to
   Drive each epoch, and re-running cell 4 after a disconnect resumes from `last.pt`. If final weights
   already exist, cell 4 skips training unless `RETRAIN = True`.
4. **Reported figures.** The metrics printed in cell 5 are measured on the internal validation split
   (10% of the training set), not on the project evaluation set.


In [ ]:
#@title 1 - Environment, paths, configuration
%pip install -q ultralytics pycocotools

import os, sys, json, glob, time, random, shutil
import numpy as np, torch

if 'google.colab' in sys.modules:
    from google.colab import drive; drive.mount('/content/drive')
    _cands = ['/content/drive/MyDrive/LISN_Project', '/content/drive/MyDrive/LISN']
    DRIVE_ROOT = next((p for p in _cands if os.path.exists(p)), _cands[0])
else:
    DRIVE_ROOT = os.environ.get('LISN_ROOT', './LISN_Project')

FLIR      = f"{DRIVE_ROOT}/datasets/flir_adas/FLIR_ADAS_1_3"
TEST_HR   = f"{DRIVE_ROOT}/datasets/IR_SR/test/HR"      # evaluation set of the comparison notebook
YOLO_DS   = "/content/yolo_thermal"                     # local SSD; Drive I/O is inadequate for training
YOLO_RUNS = f"{DRIVE_ROOT}/yolo_thermal_runs"
CKPT_OUT  = f"{DRIVE_ROOT}/checkpoints"
ASSETS    = f"{DRIVE_ROOT}/presentation_assets"

CLASS_NAMES = ['person', 'bicycle', 'car', 'dog']       # matches the comparison notebook
SEED = 123
RETRAIN = False                                         # True forces retraining even if weights exist

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
for d in (YOLO_RUNS, CKPT_OUT, ASSETS): os.makedirs(d, exist_ok=True)

print(f"DRIVE_ROOT = {DRIVE_ROOT}")
print(f"GPU: {torch.cuda.get_device_properties(0).name}" if torch.cuda.is_available()
      else "No GPU - select a GPU runtime before training")
for n, p in [('FLIR train images', f"{FLIR}/train/thermal_8_bit"),
             ('FLIR train annotations', f"{FLIR}/train/thermal_annotations.json"),
             ('evaluation-set HR', TEST_HR)]:
    print(f"  {'OK     ' if os.path.exists(p) else 'MISSING'} {n}: {p}")


In [ ]:
#@title 2 - Build the detection dataset (COCO to YOLO), with leakage check
import cv2

img_dir = f"{FLIR}/train/thermal_8_bit"
assert os.path.isdir(img_dir), f"Not found: {img_dir}"
assert 'Annotated' not in img_dir
all_files = {f: os.path.join(img_dir, f) for f in os.listdir(img_dir)}
print(f"Source directory: {img_dir} ({len(all_files)} files)")

raw = json.load(open(f"{FLIR}/train/thermal_annotations.json"))
name_by_cat = {c['id']: c['name'].lower().strip() for c in raw['categories']}
cat2idx = {cid: CLASS_NAMES.index(n) for cid, n in name_by_cat.items() if n in CLASS_NAMES}
assert cat2idx, "No annotation category matched the configured class names"
img_meta = {im['id']: im for im in raw['images']}
by_img = {}
for a in raw['annotations']:
    if a['category_id'] in cat2idx:
        by_img.setdefault(a['image_id'], []).append(a)
print(f"Annotations: {len(raw['images'])} images, "
      f"{sum(len(v) for v in by_img.values())} boxes in the configured classes")

# Leakage check: no training image may appear in the comparison notebook's evaluation set
eval_stems = {os.path.splitext(f)[0] for f in os.listdir(TEST_HR) if f.lower().endswith('.png')}
train_stems = {os.path.splitext(os.path.basename(im['file_name']))[0] for im in raw['images']}
overlap = train_stems & eval_stems
assert not overlap, f"Train/evaluation overlap of {len(overlap)} images, e.g. {sorted(overlap)[:5]}"
print(f"Leakage check: train={len(train_stems)} evaluation={len(eval_stems)} overlap=0")

# Internal 90/10 split, drawn from the training set only
usable = sorted(i for i in by_img if i in img_meta)
_rng = random.Random(SEED); _rng.shuffle(usable)
n_val = max(1, int(0.10 * len(usable)))
split = {'val': set(usable[:n_val]), 'train': set(usable[n_val:])}
print(f"Internal split: train={len(split['train'])} val={len(split['val'])}")

if os.path.exists(YOLO_DS): shutil.rmtree(YOLO_DS)
for s in ('train', 'val'):
    os.makedirs(f"{YOLO_DS}/images/{s}"); os.makedirs(f"{YOLO_DS}/labels/{s}")

stats = {'train': 0, 'val': 0, 'boxes': 0, 'skipped': 0}
for s in ('train', 'val'):
    for iid in sorted(split[s]):
        im = img_meta[iid]
        base = os.path.basename(im['file_name'])
        src = all_files.get(base)
        if src is None: stats['skipped'] += 1; continue
        W, H = im.get('width'), im.get('height')
        if not W or not H:
            g = cv2.imread(src, cv2.IMREAD_GRAYSCALE)
            if g is None: stats['skipped'] += 1; continue
            H, W = g.shape
        lines = []
        for a in by_img[iid]:
            x, y, w, h = a['bbox']
            x1, y1 = max(0., x), max(0., y)
            x2, y2 = min(float(W), x + w), min(float(H), y + h)     # clip to frame
            if x2 - x1 < 2 or y2 - y1 < 2: continue
            lines.append(f"{cat2idx[a['category_id']]} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} "
                         f"{(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
        if not lines: stats['skipped'] += 1; continue
        stem, ext = os.path.splitext(base)
        shutil.copy2(src, f"{YOLO_DS}/images/{s}/{stem}{ext}")
        open(f"{YOLO_DS}/labels/{s}/{stem}.txt", 'w').write("\n".join(lines) + "\n")
        stats[s] += 1; stats['boxes'] += len(lines)
        if (stats['train'] + stats['val']) % 1000 == 0:
            print(f"  {stats['train'] + stats['val']} images copied")

DATA_YAML = f"{YOLO_DS}/data.yaml"
open(DATA_YAML, 'w').write(f"path: {YOLO_DS}\ntrain: images/train\nval: images/val\n"
                           f"nc: {len(CLASS_NAMES)}\nnames: {CLASS_NAMES}\n")
print(f"Dataset written: {stats}")
assert stats['train'] > 1000


In [ ]:
#@title 3 - Visual verification of labels (inspect before training)
import matplotlib.pyplot as plt
sample = sorted(os.listdir(f"{YOLO_DS}/images/train"))[:2]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, fn in zip(axes, sample):
    img = cv2.cvtColor(cv2.imread(f"{YOLO_DS}/images/train/{fn}", cv2.IMREAD_GRAYSCALE),
                       cv2.COLOR_GRAY2RGB)
    H, W = img.shape[:2]
    for ln in open(f"{YOLO_DS}/labels/train/{os.path.splitext(fn)[0]}.txt"):
        c, cx, cy, bw, bh = ln.split()
        cx, cy, bw, bh = float(cx)*W, float(cy)*H, float(bw)*W, float(bh)*H
        x1, y1 = int(cx - bw/2), int(cy - bh/2)
        cv2.rectangle(img, (x1, y1), (int(cx + bw/2), int(cy + bh/2)), (80, 255, 80), 2)
        cv2.putText(img, CLASS_NAMES[int(c)], (x1, max(12, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (80, 255, 80), 1)
    ax.imshow(img); ax.axis('off'); ax.set_title(fn, fontsize=9)
fig.suptitle('Verification: images must be clean (no pre-drawn boxes); '
             'green boxes must align with objects')
fig.tight_layout(); plt.show()
print("If boxes are misplaced, or the images already contain drawn rectangles, "
      "stop and fix the source before training.")


In [ ]:
#@title 4 - Training (about 1.7 h on a T4; resumes after a disconnect)
from ultralytics import YOLO

RUN_NAME = 'flir_thermal_y8n'
FINAL_WEIGHTS = f"{CKPT_OUT}/yolov8n_flir_thermal.pt"
_best = f"{YOLO_RUNS}/{RUN_NAME}/weights/best.pt"
_last = f"{YOLO_RUNS}/{RUN_NAME}/weights/last.pt"

if os.path.exists(FINAL_WEIGHTS) and not RETRAIN:
    print(f"Final weights already exist: {FINAL_WEIGHTS}\n"
          f"Set RETRAIN = True in cell 1 to train again.")
else:
    if RETRAIN and os.path.exists(f"{YOLO_RUNS}/{RUN_NAME}"):
        shutil.rmtree(f"{YOLO_RUNS}/{RUN_NAME}")
        print("Previous run directory removed")
    _resume = os.path.exists(_last) and not os.path.exists(_best)
    print(f"{'Resuming' if _resume else 'Starting'} training")
    model = YOLO(_last) if _resume else YOLO('yolov8n.pt')   # initialise from COCO weights
    model.train(
        data=DATA_YAML,
        epochs=50, imgsz=640, batch=16,
        device=0, workers=2, seed=SEED,
        patience=10,
        project=YOLO_RUNS, name=RUN_NAME, exist_ok=True,
        resume=_resume, cache=False, val=True, plots=True, verbose=True,
    )
    print("Training complete")


In [ ]:
#@title 5 - Save weights and write the training report
from ultralytics import YOLO

_best = f"{YOLO_RUNS}/{RUN_NAME}/weights/best.pt"
FINAL_WEIGHTS = f"{CKPT_OUT}/yolov8n_flir_thermal.pt"
if os.path.exists(_best):
    shutil.copy2(_best, FINAL_WEIGHTS)
assert os.path.exists(FINAL_WEIGHTS), "No trained weights found - run cell 4"
print(f"Weights: {FINAL_WEIGHTS} ({os.path.getsize(FINAL_WEIGHTS)/1e6:.1f} MB)")

m = YOLO(FINAL_WEIGHTS).val(data=DATA_YAML, imgsz=640, device=0, verbose=False)
report = {
    'weights': FINAL_WEIGHTS,
    'trained_on': 'FLIR train split only (90% internal train, 10% internal val); '
                  'zero overlap with the 1,366-image evaluation set',
    'source_images': 'train/thermal_8_bit (clean frames)',
    'internal_val_mAP50_95': float(m.box.map),
    'internal_val_mAP50': float(m.box.map50),
    'per_class_mAP50_95': {CLASS_NAMES[i]: float(v) for i, v in enumerate(m.box.maps)},
    'classes': CLASS_NAMES, 'seed': SEED, 'epochs': 50,
    'date': time.strftime('%Y-%m-%d'),
}
tmp = f"{ASSETS}/yolo_thermal_report.json.tmp"
with open(tmp, 'w') as f: json.dump(report, f, indent=1)
os.replace(tmp, f"{ASSETS}/yolo_thermal_report.json")
print(json.dumps(report, indent=1))
print("\nFigures above are measured on the internal validation split, "
      "not on the project evaluation set.")

# Plausibility check on one evaluation image (read-only; not used for training)
_eval0 = sorted(os.listdir(TEST_HR))[0]
r = YOLO(FINAL_WEIGHTS).predict(f"{TEST_HR}/{_eval0}", conf=0.25, verbose=False)[0]
print(f"Detections on one evaluation HR image: {len(r.boxes)} "
      f"(zero on a typical street scene indicates a defective model)")


---
## Where the output is used

`IR_SR_model_comparison_FINAL_v2.ipynb`, Section 11, loads
`checkpoints/yolov8n_flir_thermal.pt` and repeats the source comparison with this detector alongside the
frozen COCO detector.

If Section 11 there reports an implausibly low HR score (below 0.15 mAP), the usual cause is training on
`Annotated_thermal_8_bit` by mistake, or stale prediction caches from a previous defective model - delete
`cache_compare_*/yolo_thermal_preds_*.json` on Drive and re-run Section 11.
